# 13-F Institutional Ownership Research

Query notebook for institutional ownership analysis. Each cell has a configurable input variable at the top.

In [1]:
# === SETUP CELL ===
# Imports, DuckDB connection, and helpers

import duckdb
import pandas as pd
from datetime import datetime
import os

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

DB_PATH = os.path.join('..', 'data', '13f.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

today = datetime.now().strftime('%Y%m%d')

def fmt_dollars(val):
    """Format dollar values as $1.2B, $340M, etc."""
    if pd.isna(val) or val == 0:
        return '$0'
    abs_val = abs(val)
    sign = '-' if val < 0 else ''
    if abs_val >= 1e12:
        return f'{sign}${abs_val/1e12:.1f}T'
    elif abs_val >= 1e9:
        return f'{sign}${abs_val/1e9:.1f}B'
    elif abs_val >= 1e6:
        return f'{sign}${abs_val/1e6:.0f}M'
    elif abs_val >= 1e3:
        return f'{sign}${abs_val/1e3:.0f}K'
    else:
        return f'{sign}${abs_val:,.0f}'

def fmt_shares(val):
    """Format share counts as 94.2M, 5.5K, etc."""
    if pd.isna(val) or val == 0:
        return '0'
    abs_val = abs(val)
    sign = '-' if val < 0 else ''
    if abs_val >= 1e9:
        return f'{sign}{abs_val/1e9:.1f}B'
    elif abs_val >= 1e6:
        return f'{sign}{abs_val/1e6:.1f}M'
    elif abs_val >= 1e3:
        return f'{sign}{abs_val/1e3:.1f}K'
    else:
        return f'{sign}{abs_val:,.0f}'

def fmt_pct(val):
    if pd.isna(val):
        return '—'
    return f'{val:.2f}%'

# Verify tables
tables = con.execute("SHOW TABLES").fetchdf()
print('Available tables:', tables['name'].tolist())
print(f'\nConnection: {DB_PATH}')
print(f'Date: {today}')

Available tables: ['adv_managers', 'cik_crd_direct', 'cik_crd_links', 'filings', 'filings_deduped', 'fund_name_map', 'holdings', 'managers', 'market_data', 'other_managers', 'parent_bridge', 'raw_coverpage', 'raw_infotable', 'raw_submissions', 'securities']

Connection: ../data/13f.duckdb
Date: 20260401


---
## QUERY 1 — Current Shareholder Register (Two-Level Hierarchy)
Shows top institutional parents with subsidiary fund detail.
Value = shares × today's live price.

In [2]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'  # Change to any ticker
# ==============================

# Get CUSIP for ticker
cusip_row = con.execute(f"SELECT cusip FROM securities WHERE ticker = '{TICKER}' LIMIT 1").fetchone()
if cusip_row:
    CUSIP = cusip_row[0]
else:
    CUSIP = None
    print(f'Ticker {TICKER} not found in securities table. Searching by issuer name...')

# Parent-level aggregation
q1_parents = con.execute(f"""
    WITH by_fund AS (
        SELECT
            COALESCE(h.inst_parent_name, h.manager_name) as parent_name,
            h.fund_name,
            h.cik,
            COALESCE(h.manager_type, 'unknown') as type,
            h.market_value_live,
            h.shares,
            h.pct_of_float
        FROM holdings h
        WHERE h.quarter = '2025Q4'
          AND (h.ticker = '{TICKER}' OR h.cusip = '{CUSIP}')
    )
    SELECT
        parent_name,
        MAX(type) as type,
        SUM(market_value_live) as total_value_live,
        SUM(shares) as total_shares,
        SUM(pct_of_float) as pct_float,
        COUNT(*) as num_funds
    FROM by_fund
    GROUP BY parent_name
    ORDER BY total_value_live DESC NULLS LAST
    LIMIT 15
""").fetchdf()

print(f'=== SHAREHOLDER REGISTER: {TICKER} ===')
print(f'Note: Share counts as of Q4 2025 13F filing. Valued at today\'s price.\n')

print(f'{"Rank":>4} {"Institution Parent":<40} {"Total Value":>12} {"Shares":>10} {"% Float":>8} {"Type":>10}')
print('-' * 90)

for rank, (_, parent) in enumerate(q1_parents.iterrows(), 1):
    pname = parent['parent_name']
    print(f"{rank:>4} {str(pname)[:40]:<40} {fmt_dollars(parent['total_value_live']):>12} "
          f"{fmt_shares(parent['total_shares']):>10} "
          f"{fmt_pct(parent['pct_float']):>8} {parent['type']:>10}")

    # Get individual fund-level holdings under this parent, grouped by fund_name
    subs = con.execute(f"""
        SELECT
            h.fund_name,
            COALESCE(h.manager_type, 'unknown') as type,
            SUM(h.market_value_live) as value_live,
            SUM(h.shares) as shares,
            SUM(h.pct_of_float) as pct_of_float
        FROM holdings h
        WHERE h.quarter = '2025Q4'
          AND (h.ticker = '{TICKER}' OR h.cusip = '{CUSIP}')
          AND COALESCE(h.inst_parent_name, h.manager_name) = '{pname.replace(chr(39), chr(39)+chr(39))}'
        GROUP BY h.fund_name, type
        ORDER BY value_live DESC NULLS LAST
        LIMIT 5
    """).fetchdf()

    for _, sub in subs.iterrows():
        print(f"     {'':>1}\u21b3 {str(sub['fund_name'])[:38]:<38} {fmt_dollars(sub['value_live']):>12} "
              f"{fmt_shares(sub['shares']):>10} "
              f"{fmt_pct(sub['pct_of_float']):>8} {sub['type']:>10}")
    print()

# Export
q1_export = con.execute(f"""
    SELECT
        COALESCE(h.inst_parent_name, h.manager_name) as parent_name,
        h.fund_name,
        h.cik,
        COALESCE(h.manager_type, 'unknown') as type,
        h.market_value_live as value_live,
        h.shares,
        h.pct_of_float,
        h.pct_of_portfolio
    FROM holdings h
    WHERE h.quarter = '2025Q4'
      AND (h.ticker = '{TICKER}' OR h.cusip = '{CUSIP}')
    ORDER BY value_live DESC NULLS LAST
""").fetchdf()
q1_export.to_excel(f'../outputs/query1_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query1_{TICKER}_{today}.xlsx')

=== SHAREHOLDER REGISTER: AR ===
Note: Share counts as of Q4 2025 13F filing. Valued at today's price.

Rank Institution Parent                        Total Value     Shares  % Float       Type
------------------------------------------------------------------------------------------
   1 Vanguard Group                                  $1.2B      28.7M    9.86%    passive
      ↳ VANGUARD GROUP INC                            $1.1B      26.4M    9.08%    passive
      ↳ VANGUARD FIDUCIARY TRUST CO                    $76M       1.8M    0.61%    passive
      ↳ VANGUARD ASSET MANAGEMENT, Ltd                  $9M     219.8K    0.08%    passive
      ↳ Vanguard Global Advisers, LLC                   $5M     123.6K    0.04%    passive
      ↳ Vanguard Investments Australia, Ltd.            $4M     102.0K    0.04%    passive

   2 BlackRock / iShares                             $1.1B      26.1M    8.98%    passive
      ↳ BlackRock Fund Advisors                       $651M      15.3M    5.28%

---
## QUERY 2 — 4-Quarter Ownership Change (Q1 2025 vs Q4 2025)
Shows share changes at fund level under parent grouping.

In [3]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

cusip_row = con.execute(f"SELECT cusip FROM securities WHERE ticker = '{TICKER}' LIMIT 1").fetchone()
CUSIP = cusip_row[0] if cusip_row else ''

# Get top 15 parents by Q4 position (same universe as Query 1)
top_parents = con.execute(f"""
    SELECT COALESCE(inst_parent_name, manager_name) as parent_name,
           SUM(market_value_live) as parent_val
    FROM holdings
    WHERE quarter = '2025Q4' AND (ticker = '{TICKER}' OR cusip = '{CUSIP}')
    GROUP BY parent_name
    ORDER BY parent_val DESC NULLS LAST
    LIMIT 15
""").fetchdf()['parent_name'].tolist()
parent_filter = ",".join(f"''{p.replace(chr(39), chr(39)+chr(39))}''" for p in top_parents)

# Aggregate to CIK level per quarter BEFORE joining — prevents cartesian product
q2 = con.execute(f"""
    WITH q1_agg AS (
        SELECT cik, manager_name,
               COALESCE(inst_parent_name, manager_name) as parent_name,
               MAX(manager_type) as manager_type,
               SUM(shares) as q1_shares
        FROM holdings
        WHERE quarter = '2025Q1' AND (ticker = '{TICKER}' OR cusip = '{CUSIP}')
        GROUP BY cik, manager_name, parent_name
    ),
    q4_agg AS (
        SELECT cik, manager_name,
               COALESCE(inst_parent_name, manager_name) as parent_name,
               MAX(manager_type) as manager_type,
               SUM(shares) as q4_shares
        FROM holdings
        WHERE quarter = '2025Q4' AND (ticker = '{TICKER}' OR cusip = '{CUSIP}')
        GROUP BY cik, manager_name, parent_name
    ),
    combined AS (
        SELECT
            COALESCE(q4.parent_name, q1.parent_name) as parent_name,
            COALESCE(q4.manager_name, q1.manager_name) as fund_name,
            COALESCE(q4.cik, q1.cik) as cik,
            COALESCE(q4.manager_type, q1.manager_type, 'unknown') as type,
            q1.q1_shares,
            q4.q4_shares,
            COALESCE(q4.q4_shares, 0) - COALESCE(q1.q1_shares, 0) as change_shares,
            CASE
                WHEN q1.q1_shares > 0 AND q4.q4_shares IS NOT NULL
                THEN ROUND((q4.q4_shares - q1.q1_shares) * 100.0 / q1.q1_shares, 1)
                WHEN q1.q1_shares IS NULL THEN NULL
                ELSE -100.0
            END as change_pct,
            CASE WHEN q1.q1_shares IS NULL THEN true ELSE false END as is_entry,
            CASE WHEN q4.q4_shares IS NULL THEN true ELSE false END as is_exit
        FROM q4_agg q4
        FULL OUTER JOIN q1_agg q1 ON q4.cik = q1.cik
    )
    SELECT * FROM combined
    ORDER BY parent_name, ABS(change_shares) DESC
""").fetchdf()

print(f'=== 4-QUARTER OWNERSHIP CHANGE: {TICKER} ===')
print(f'Q1 2025 vs Q4 2025. * = active fund (conviction move, not index rebalancing)\n')

# Filter to top 15 parents
q2_top = q2[q2['parent_name'].isin(top_parents) & ~q2['is_entry'] & ~q2['is_exit']]

print(f'{"Institution / Fund":<45} {"Q1 Shares":>12} {"Q4 Shares":>12} {"Change":>12} {"Chg%":>8} {"Type":>10}')
print('-' * 105)

current_parent = None
for _, row in q2_top.iterrows():
    if row['parent_name'] != current_parent:
        current_parent = row['parent_name']
        # Parent summary
        parent_rows = q2_top[q2_top['parent_name'] == current_parent]
        p_q1 = parent_rows['q1_shares'].sum()
        p_q4 = parent_rows['q4_shares'].sum()
        p_chg = p_q4 - p_q1
        p_pct = f"{p_chg/p_q1*100:.1f}%" if p_q1 > 0 else "NEW"
        print(f"\n  {current_parent}")
        print(f"  {'(parent total)':<43} {fmt_shares(p_q1):>12} {fmt_shares(p_q4):>12} {fmt_shares(p_chg):>12} {p_pct:>8}")

    flag = '*' if row['type'] not in ('passive', 'unknown', None) else ' '
    q1s = fmt_shares(row['q1_shares']) if pd.notna(row['q1_shares']) else '—'
    q4s = fmt_shares(row['q4_shares']) if pd.notna(row['q4_shares']) else '—'
    chg = fmt_shares(row['change_shares'])
    pct = f"{row['change_pct']:.1f}%" if pd.notna(row['change_pct']) else '—'
    print(f"    {flag} {str(row['fund_name'])[:42]:<42} {q1s:>12} {q4s:>12} {chg:>12} {pct:>8} {str(row['type']):>10}")

# Entries (new in Q4, not in Q1) — over 100K shares
entries = q2[q2['is_entry'] & (q2['q4_shares'] >= 100000)].sort_values('q4_shares', ascending=False)
if len(entries) > 0:
    print(f'\n--- NEW ENTRIES (>100K shares): {len(entries)} ---')
    for _, e in entries.head(15).iterrows():
        flag = '*' if e['type'] not in ('passive', 'unknown', None) else ' '
        print(f"  {flag} {str(e['fund_name'])[:42]:<42} {fmt_shares(e['q4_shares']):>12}  {e['type']}")

# Exits (in Q1, gone in Q4) — over 100K shares
exits = q2[q2['is_exit'] & (q2['q1_shares'] >= 100000)].sort_values('q1_shares', ascending=False)
if len(exits) > 0:
    print(f'\n--- EXITS (>100K shares): {len(exits)} ---')
    for _, e in exits.head(15).iterrows():
        flag = '*' if e['type'] not in ('passive', 'unknown', None) else ' '
        print(f"  {flag} {str(e['fund_name'])[:42]:<42} {fmt_shares(e['q1_shares']):>12} (Q1)  {e['type']}")

q2.to_excel(f'../outputs/query2_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query2_{TICKER}_{today}.xlsx')

=== 4-QUARTER OWNERSHIP CHANGE: AR ===
Q1 2025 vs Q4 2025. * = active fund (conviction move, not index rebalancing)

Institution / Fund                               Q1 Shares    Q4 Shares       Change     Chg%       Type
---------------------------------------------------------------------------------------------------------

  AQR Capital
  (parent total)                                      1.3M         4.7M         3.4M   269.7%
    * AQR CAPITAL MANAGEMENT LLC                         1.3M         4.7M         3.4M   269.7% quantitative

  BlackRock / iShares
  (parent total)                                     28.2M        26.1M        -2.2M    -7.6%
      BlackRock, Inc.                                   28.2M        26.1M        -2.2M    -7.6%    passive

  DE Shaw
  (parent total)                                      3.1M         4.1M         1.1M    34.4%
    * D. E. Shaw & Co., Inc.                             3.1M         4.1M         1.1M    34.4% quantitative

  Dimensiona

---
## QUERY 3 — Active Holder Market Cap Analysis
Top 15 active holders with position sizing context.

In [4]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

cusip_row = con.execute(f"SELECT cusip FROM securities WHERE ticker = '{TICKER}' LIMIT 1").fetchone()
CUSIP = cusip_row[0] if cusip_row else ''

# Get target company market cap
mktcap_row = con.execute(f"SELECT market_cap FROM market_data WHERE ticker = '{TICKER}'").fetchone()
target_mktcap = mktcap_row[0] if mktcap_row else None

# Check if fund_holdings table exists (N-PORT data)
has_fund_holdings = False
try:
    con.execute("SELECT 1 FROM fund_holdings LIMIT 1")
    has_fund_holdings = True
except:
    pass

# Core query: aggregate to one row per CIK, then compute percentile
q3 = con.execute(f"""
    WITH cik_agg AS (
        SELECT
            h.cik,
            MAX(h.manager_name) as manager_name,
            MAX(h.manager_type) as manager_type,
            SUM(h.market_value_live) as position_value,
            SUM(h.shares) as shares,
            MAX(h.pct_of_portfolio) as pct_of_portfolio,
            SUM(h.pct_of_float) as pct_of_float
        FROM holdings h
        WHERE h.quarter = '2025Q4'
          AND (h.ticker = '{TICKER}' OR h.cusip = '{CUSIP}')
          AND h.manager_type IN ('active', 'hedge_fund', 'activist', 'quantitative')
        GROUP BY h.cik
        ORDER BY position_value DESC NULLS LAST
        LIMIT 15
    ),
    with_percentile AS (
        SELECT
            ca.*,
            (
                SELECT COUNT(*)
                FROM holdings h2
                INNER JOIN market_data m2 ON h2.ticker = m2.ticker
                WHERE h2.cik = ca.cik AND h2.quarter = '2025Q4'
                  AND h2.security_type_inferred IN ('equity', 'etf')
                  AND m2.market_cap IS NOT NULL AND m2.market_cap > 0
                  AND m2.market_cap <= {target_mktcap if target_mktcap else 0}
            ) as holdings_below,
            (
                SELECT COUNT(*)
                FROM holdings h2
                INNER JOIN market_data m2 ON h2.ticker = m2.ticker
                WHERE h2.cik = ca.cik AND h2.quarter = '2025Q4'
                  AND h2.security_type_inferred IN ('equity', 'etf')
                  AND m2.market_cap IS NOT NULL AND m2.market_cap > 0
            ) as total_with_mktcap
        FROM cik_agg ca
    )
    SELECT
        manager_name as "Active Holder",
        position_value,
        pct_of_portfolio,
        pct_of_float,
        CASE WHEN total_with_mktcap > 0
             THEN ROUND(holdings_below * 100.0 / total_with_mktcap, 1)
             ELSE NULL END as mktcap_percentile,
        manager_type,
        '13F estimate' as source
    FROM with_percentile
    ORDER BY position_value DESC NULLS LAST
""").fetchdf()

print(f'=== ACTIVE HOLDER MARKET CAP ANALYSIS: {TICKER} ===')
print(f'Target market cap: {fmt_dollars(target_mktcap) if target_mktcap else "N/A"}\n')

print(f'{"Active Holder":<40} {"Position Value":>14} {"% Portfolio":>12} '
      f'{"% of " + TICKER + " Float":>14} {"MktCap Pctile":>14} {"Type":>12} {"Source":>14}')
print('-' * 125)

for _, row in q3.iterrows():
    print(f"{str(row['Active Holder'])[:40]:<40} "
          f"{fmt_dollars(row['position_value']):>14} "
          f"{fmt_pct(row['pct_of_portfolio']):>12} "
          f"{fmt_pct(row['pct_of_float']):>14} "
          f"{fmt_pct(row['mktcap_percentile']):>14} "
          f"{str(row['manager_type']):>12} "
          f"{str(row['source']):>14}")

q3.to_excel(f'../outputs/query3_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query3_{TICKER}_{today}.xlsx')

=== ACTIVE HOLDER MARKET CAP ANALYSIS: AR ===
Target market cap: $13.1B

Active Holder                            Position Value  % Portfolio  % of AR Float  MktCap Pctile         Type         Source
-----------------------------------------------------------------------------------------------------------------------------
FMR LLC                                           $1.0B        0.03%          8.14%         65.30%       active   13F estimate
WELLINGTON MANAGEMENT GROUP LLP                   $480M        0.06%          3.89%         53.70%       active   13F estimate
AQR CAPITAL MANAGEMENT LLC                        $199M        0.08%          1.61%         70.70% quantitative   13F estimate
RAYMOND JAMES FINANCIAL INC                       $185M        0.02%          1.50%         62.00%       active   13F estimate
D. E. Shaw & Co., Inc.                            $176M        0.05%          1.42%         63.00% quantitative   13F estimate
MASSACHUSETTS FINANCIAL SERVICES CO /MA

---
## QUERY 4 — Passive vs Active Ownership Split
Shows what fraction of institutional ownership is passive (index funds) vs active.

In [5]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q4 = con.execute(f"""
    SELECT
        CASE
            WHEN manager_type = 'passive' THEN 'Passive (Index)'
            WHEN manager_type = 'activist' THEN 'Activist'
            WHEN manager_type IN ('active', 'hedge_fund', 'quantitative') THEN 'Active'
            ELSE 'Other/Unknown'
        END as category,
        COUNT(DISTINCT cik) as num_holders,
        SUM(shares) as total_shares,
        SUM(market_value_live) as total_value,
        SUM(pct_of_float) as total_pct_float
    FROM holdings
    WHERE quarter = '2025Q4' AND ticker = '{TICKER}'
    GROUP BY category
    ORDER BY total_value DESC NULLS LAST
""").fetchdf()

print(f'=== PASSIVE vs ACTIVE SPLIT: {TICKER} (Q4 2025) ===\n')
grand_total = q4['total_value'].sum()
for _, row in q4.iterrows():
    pct_of_inst = row['total_value'] / grand_total * 100 if grand_total > 0 else 0
    print(f"  {row['category']:<20} {row['num_holders']:>5} holders  "
          f"{fmt_dollars(row['total_value']):>12}  {pct_of_inst:>5.1f}% of inst. ownership")

q4.to_excel(f'../outputs/query4_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query4_{TICKER}_{today}.xlsx')

=== PASSIVE vs ACTIVE SPLIT: AR (Q4 2025) ===

  Other/Unknown          469 holders         $4.7B   39.6% of inst. ownership
  Passive (Index)         14 holders         $3.9B   32.5% of inst. ownership
  Active                 103 holders         $3.3B   27.9% of inst. ownership

Exported to outputs/query4_AR_20260401.xlsx


---
## QUERY 5 — Quarterly Share Change Heatmap
Shows Q-over-Q share changes for top holders across all 4 quarters.

In [6]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q5 = con.execute(f"""
    WITH pivoted AS (
        SELECT
            COALESCE(inst_parent_name, manager_name) as holder,
            manager_type,
            SUM(CASE WHEN quarter='2025Q1' THEN shares END) as q1_shares,
            SUM(CASE WHEN quarter='2025Q2' THEN shares END) as q2_shares,
            SUM(CASE WHEN quarter='2025Q3' THEN shares END) as q3_shares,
            SUM(CASE WHEN quarter='2025Q4' THEN shares END) as q4_shares
        FROM holdings
        WHERE ticker = '{TICKER}'
        GROUP BY holder, manager_type
    )
    SELECT *,
        q2_shares - q1_shares as q1_to_q2,
        q3_shares - q2_shares as q2_to_q3,
        q4_shares - q3_shares as q3_to_q4,
        q4_shares - q1_shares as full_year_change
    FROM pivoted
    WHERE q4_shares IS NOT NULL
    ORDER BY q4_shares DESC
    LIMIT 25
""").fetchdf()

print(f'=== QUARTERLY SHARE CHANGES: {TICKER} ===\n')
print(f'{"Holder":<35} {"Q1":>10} {"Q2":>10} {"Q3":>10} {"Q4":>10} {"Q1→Q2":>10} {"Q2→Q3":>10} {"Q3→Q4":>10} {"Full Yr":>10}')
print('-' * 125)
for _, r in q5.iterrows():
    print(f"{str(r['holder'])[:35]:<35} {fmt_shares(r['q1_shares']):>10} {fmt_shares(r['q2_shares']):>10} "
          f"{fmt_shares(r['q3_shares']):>10} {fmt_shares(r['q4_shares']):>10} "
          f"{fmt_shares(r['q1_to_q2']):>10} {fmt_shares(r['q2_to_q3']):>10} "
          f"{fmt_shares(r['q3_to_q4']):>10} {fmt_shares(r['full_year_change']):>10}")

q5.to_excel(f'../outputs/query5_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query5_{TICKER}_{today}.xlsx')

=== QUARTERLY SHARE CHANGES: AR ===

Holder                                      Q1         Q2         Q3         Q4      Q1→Q2      Q2→Q3      Q3→Q4    Full Yr
-----------------------------------------------------------------------------------------------------------------------------
Vanguard Group                           29.5M      29.8M      29.5M      28.7M     352.5K    -304.0K    -867.0K    -818.4K
BlackRock / iShares                      28.2M      29.9M      28.2M      26.1M       1.7M      -1.7M      -2.1M      -2.2M
Fidelity / FMR                           25.7M      25.0M      25.1M      23.7M    -681.6K     132.1K      -1.5M      -2.0M
Wellington Management                        0      19.3M      18.7M      11.3M          0    -608.3K      -7.3M          0
State Street / SSGA                      10.4M      10.1M      10.2M       9.9M    -321.6K     116.5K    -285.8K    -490.8K
Sourcerock Group LLC                      7.4M       8.2M       7.8M       9.2M     783.1K   

---
## QUERY 6 — Activist Ownership Tracker
All activist positions in a given stock across all quarters.

In [7]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q6 = con.execute(f"""
    SELECT
        manager_name,
        quarter,
        shares,
        market_value_usd,
        market_value_live,
        pct_of_portfolio,
        pct_of_float
    FROM holdings
    WHERE ticker = '{TICKER}' AND is_activist = true
    ORDER BY manager_name, quarter
""").fetchdf()

print(f'=== ACTIVIST POSITIONS: {TICKER} ===\n')
if len(q6) == 0:
    print('No activist positions found.')
else:
    for name in q6['manager_name'].unique():
        sub = q6[q6['manager_name'] == name]
        print(f'\n  {name}:')
        for _, r in sub.iterrows():
            print(f"    {r['quarter']}  {fmt_shares(r['shares']):>10}  "
                  f"{fmt_dollars(r['market_value_usd']):>12} (filed)  "
                  f"{fmt_pct(r['pct_of_portfolio']):>8} of portfolio  "
                  f"{fmt_pct(r['pct_of_float']):>8} of float")

q6.to_excel(f'../outputs/query6_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query6_{TICKER}_{today}.xlsx')

=== ACTIVIST POSITIONS: AR ===


  ENGINE NO. 1 LLC:
    2025Q1      138.0K         $5.6B (filed)     7.12% of portfolio     0.05% of float

Exported to outputs/query6_AR_20260401.xlsx


---
## QUERY 7 — Top Portfolio Positions for a Given Manager (CIK)
Full portfolio view for a single institutional filer.

In [8]:
# ===== CHANGE CIK HERE (Berkshire Hathaway) =====
CIK = '0001067983'
# ================================================

q7 = con.execute(f"""
    SELECT
        h.ticker,
        h.issuer_name,
        h.cusip,
        h.shares,
        h.market_value_usd,
        h.market_value_live,
        h.pct_of_portfolio,
        h.pct_of_float,
        m.market_cap
    FROM holdings h
    LEFT JOIN market_data m ON h.ticker = m.ticker
    WHERE h.cik = '{CIK}' AND h.quarter = '2025Q4'
    ORDER BY h.market_value_live DESC NULLS LAST
    LIMIT 30
""").fetchdf()

mgr_name = con.execute(f"SELECT manager_name FROM filings_deduped WHERE cik = '{CIK}' LIMIT 1").fetchone()
name = mgr_name[0] if mgr_name else CIK

print(f'=== TOP HOLDINGS: {name} (Q4 2025) ===\n')
print(f'{"#":>3} {"Ticker":<8} {"Issuer":<30} {"Shares":>12} {"Value (Live)":>14} {"% Port":>8} {"% Float":>8}')
print('-' * 90)
for i, (_, r) in enumerate(q7.iterrows(), 1):
    print(f"{i:>3} {str(r['ticker'] or '')::<8} {str(r['issuer_name'])[:30]:<30} "
          f"{fmt_shares(r['shares']):>12} {fmt_dollars(r['market_value_live']):>14} "
          f"{fmt_pct(r['pct_of_portfolio']):>8} {fmt_pct(r['pct_of_float']):>8}")

q7.to_excel(f'../outputs/query7_{CIK}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query7_{CIK}_{today}.xlsx')

=== TOP HOLDINGS: Berkshire Hathaway Inc (Q4 2025) ===

  # Ticker   Issuer                               Shares   Value (Live)   % Port  % Float
------------------------------------------------------------------------------------------
  1 AXP::::: AMERICAN EXPRESS CO                  149.1M         $45.1B   20.11%   27.95%
  2 KO:::::: COCA COLA CO                         282.7M         $21.5B    7.21%    7.30%
  3 AAPL:::: APPLE INC                             80.7M         $20.5B    8.00%    0.55%
  4 OXY::::: OCCIDENTAL PETE CORP                 264.9M         $17.2B    3.97%   26.97%
  5 AAPL:::: APPLE INC                             61.5M         $15.6B    6.10%    0.42%
  6 BAC::::: BANK AMERICA CORP                    310.8M         $15.2B    6.24%    4.35%
  7 CVX::::: CHEVRON CORP NEW                      61.6M         $12.7B    3.42%    3.32%
  8 AAPL:::: APPLE INC                             34.7M          $8.8B    3.44%    0.24%
  9 KHC::::: KRAFT HEINZ CO                

---
## QUERY 8 — Cross-Holder Overlap (Common Ownership)
Find stocks most commonly held alongside the target.

In [9]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q8 = con.execute(f"""
    WITH target_holders AS (
        SELECT DISTINCT cik
        FROM holdings
        WHERE ticker = '{TICKER}' AND quarter = '2025Q4'
    )
    SELECT
        h.ticker,
        h.issuer_name,
        COUNT(DISTINCT h.cik) as shared_holders,
        SUM(h.market_value_live) as total_value,
        (SELECT COUNT(*) FROM target_holders) as target_holders_count
    FROM holdings h
    INNER JOIN target_holders th ON h.cik = th.cik
    WHERE h.quarter = '2025Q4'
      AND h.ticker != '{TICKER}'
      AND h.ticker IS NOT NULL
    GROUP BY h.ticker, h.issuer_name
    ORDER BY shared_holders DESC
    LIMIT 20
""").fetchdf()

print(f'=== CROSS-HOLDER OVERLAP: {TICKER} (Q4 2025) ===')
print(f'Stocks most commonly held by the same institutions that hold {TICKER}\n')

for _, r in q8.iterrows():
    pct = r['shared_holders'] / r['target_holders_count'] * 100
    print(f"  {r['ticker']:<8} {str(r['issuer_name'])[:30]:<30} "
          f"{r['shared_holders']:>5} holders ({pct:.0f}%)  {fmt_dollars(r['total_value'])}")

q8.to_excel(f'../outputs/query8_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query8_{TICKER}_{today}.xlsx')

=== CROSS-HOLDER OVERLAP: AR (Q4 2025) ===
Stocks most commonly held by the same institutions that hold AR

  MSFT     MICROSOFT CORP                   438 holders (75%)  $1.8T
  AAPL     APPLE INC                        434 holders (74%)  $1.9T
  AVGO     BROADCOM INC                     422 holders (72%)  $849.7B
  TSLA     TESLA INC                        408 holders (70%)  $598.4B
  XOM      EXXON MOBIL CORP                 400 holders (68%)  $380.7B
  WMT      WALMART INC                      400 holders (68%)  $291.7B
  NFLX     NETFLIX INC                      396 holders (68%)  $251.9B
  NVDA     NVIDIA CORPORATION               394 holders (67%)  $2.2T
  GOOGL    ALPHABET INC                     394 holders (67%)  $896.7B
  META     META PLATFORMS INC               394 holders (67%)  $725.7B
  ABBV     ABBVIE INC                       393 holders (67%)  $224.9B
  AMZN     AMAZON COM INC                   392 holders (67%)  $1.0T
  CAT      CATERPILLAR INC                  390 

---
## QUERY 9 — Sector Rotation Analysis
Which sectors are the top holders of a stock overweight/underweight vs the market?

In [10]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q9 = con.execute(f"""
    WITH target_ciks AS (
        SELECT DISTINCT cik
        FROM holdings
        WHERE ticker = '{TICKER}' AND quarter = '2025Q4'
        AND manager_type IN ('active', 'hedge_fund')
    )
    SELECT
        s.sector,
        COUNT(DISTINCT h.ticker) as num_stocks,
        SUM(h.market_value_live) as sector_value,
        SUM(h.market_value_live) * 100.0 / SUM(SUM(h.market_value_live)) OVER () as pct_of_total
    FROM holdings h
    INNER JOIN target_ciks tc ON h.cik = tc.cik
    INNER JOIN securities s ON h.cusip = s.cusip
    WHERE h.quarter = '2025Q4' AND s.sector IS NOT NULL AND s.sector != ''
    GROUP BY s.sector
    ORDER BY sector_value DESC NULLS LAST
""").fetchdf()

print(f'=== SECTOR ALLOCATION: Active holders of {TICKER} (Q4 2025) ===\n')
for _, r in q9.iterrows():
    print(f"  {str(r['sector']):<30} {r['num_stocks']:>5} stocks  "
          f"{fmt_dollars(r['sector_value']):>12}  {r['pct_of_total']:.1f}%")

q9.to_excel(f'../outputs/query9_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query9_{TICKER}_{today}.xlsx')

=== SECTOR ALLOCATION: Active holders of AR (Q4 2025) ===

  Technology                        70 stocks         $1.4T  34.2%
  Communication Services            20 stocks       $477.3B  12.1%
  Financial Services                60 stocks       $435.7B  11.0%
  Healthcare                        42 stocks       $424.6B  10.8%
  Consumer Cyclical                 39 stocks       $399.7B  10.1%
  Industrials                       54 stocks       $329.4B  8.3%
  Energy                            25 stocks       $174.8B  4.4%
  Consumer Defensive                22 stocks       $132.4B  3.4%
  Utilities                         23 stocks       $110.9B  2.8%
  Basic Materials                   16 stocks        $64.2B  1.6%
  Real Estate                       14 stocks        $48.1B  1.2%

Exported to outputs/query9_AR_20260401.xlsx


---
## QUERY 10 — Largest New Positions (New Money)
Biggest new positions established in Q4 2025 (not held in Q3).

In [11]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q10 = con.execute(f"""
    SELECT
        q4.manager_name,
        q4.manager_type,
        q4.shares,
        q4.market_value_live,
        q4.pct_of_portfolio,
        q4.pct_of_float
    FROM holdings q4
    LEFT JOIN holdings q3 ON q4.cik = q3.cik AND q3.ticker = '{TICKER}' AND q3.quarter = '2025Q3'
    WHERE q4.ticker = '{TICKER}' AND q4.quarter = '2025Q4' AND q3.cik IS NULL
    ORDER BY q4.market_value_live DESC NULLS LAST
    LIMIT 20
""").fetchdf()

print(f'=== NEW POSITIONS IN {TICKER}: Q4 2025 ===\n')
for i, (_, r) in enumerate(q10.iterrows(), 1):
    print(f"  {i:>2}. {r['manager_name']:<40} {fmt_shares(r['shares']):>10}  "
          f"{fmt_dollars(r['market_value_live']):>12}  {r['manager_type']}")

q10.to_excel(f'../outputs/query10_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query10_{TICKER}_{today}.xlsx')

=== NEW POSITIONS IN AR: Q4 2025 ===

   1. Eurizon Capital SGR S.p.A.                     1.7M          $72M  None
   2. Webs Creek Capital Management LP               1.5M          $64M  None
   3. DME Capital Management, LP                   838.7K          $36M  None
   4. Orbis Allan Gray Ltd                         733.0K          $31M  None
   5. Merewether Investment Management, LP         727.7K          $31M  hedge_fund
   6. IES Holdings, Inc.                           705.8K          $30M  None
   7. Waratah Capital Advisors Ltd.                590.0K          $25M  hedge_fund
   8. NewEdge Wealth, LLC                          583.1K          $25M  None
   9. Covalis Capital LLP                          433.1K          $18M  None
  10. HATCH COVE CAPITAL, LLC                      400.0K          $17M  None
  11. Centiva Capital, LP                          400.0K          $17M  None
  12. CIBC WORLD MARKETS CORP                      375.0K          $16M  None
  13. CLOUGH C

---
## QUERY 11 — Largest Exits (Sold Entirely)
Managers who sold their entire position between Q3 and Q4 2025.

In [12]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q11 = con.execute(f"""
    SELECT
        q3.manager_name,
        q3.manager_type,
        q3.shares as q3_shares,
        q3.market_value_usd as q3_value,
        q3.pct_of_portfolio as q3_pct
    FROM holdings q3
    LEFT JOIN holdings q4 ON q3.cik = q4.cik AND q4.ticker = '{TICKER}' AND q4.quarter = '2025Q4'
    WHERE q3.ticker = '{TICKER}' AND q3.quarter = '2025Q3' AND q4.cik IS NULL
    ORDER BY q3.market_value_usd DESC
    LIMIT 20
""").fetchdf()

print(f'=== FULL EXITS FROM {TICKER}: Q3→Q4 2025 ===\n')
for i, (_, r) in enumerate(q11.iterrows(), 1):
    print(f"  {i:>2}. {r['manager_name']:<40} {fmt_shares(r['q3_shares']):>10}  "
          f"{fmt_dollars(r['q3_value']):>12} (Q3 value)  {r['manager_type']}")

q11.to_excel(f'../outputs/query11_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query11_{TICKER}_{today}.xlsx')

=== FULL EXITS FROM AR: Q3→Q4 2025 ===

   1. TEXAS PERMANENT SCHOOL FUND CORP             337.7K        $13.6B (Q3 value)  None
   2. COBALT CAPITAL MANAGEMENT, INC.              320.0K        $10.7B (Q3 value)  None
   3. WOLVERINE TRADING, LLC                       245.6K         $8.2B (Q3 value)  None
   4. MACQUARIE GROUP LTD                          236.0K         $7.9B (Q3 value)  None
   5. ExodusPoint Capital Management, LP           226.1K         $7.6B (Q3 value)  None
   6. SummitTX Capital, L.P.                       192.5K         $6.5B (Q3 value)  None
   7. Wealth High Governance Capital Ltda          180.0K         $6.0B (Q3 value)  None
   8. WOLVERINE TRADING, LLC                       174.3K         $5.8B (Q3 value)  None
   9. HSBC HOLDINGS PLC                            135.5K         $4.5B (Q3 value)  None
  10. Sciencast Management LP                      119.0K         $4.0B (Q3 value)  None
  11. Interval Partners, LP                        116.2K         $3.9

---
## QUERY 12 — Concentration Analysis
How concentrated is ownership? Top 5/10/20 holder % of float.

In [13]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q12 = con.execute(f"""
    WITH ranked AS (
        SELECT
            COALESCE(inst_parent_name, manager_name) as holder,
            SUM(pct_of_float) as total_pct_float,
            SUM(shares) as total_shares,
            ROW_NUMBER() OVER (ORDER BY SUM(pct_of_float) DESC) as rn
        FROM holdings
        WHERE ticker = '{TICKER}' AND quarter = '2025Q4' AND pct_of_float IS NOT NULL
        GROUP BY holder
    )
    SELECT
        rn as rank,
        holder,
        total_pct_float,
        total_shares,
        SUM(total_pct_float) OVER (ORDER BY rn) as cumulative_pct
    FROM ranked
    ORDER BY rn
    LIMIT 20
""").fetchdf()

print(f'=== CONCENTRATION ANALYSIS: {TICKER} (Q4 2025) ===\n')
for _, r in q12.iterrows():
    print(f"  {int(r['rank']):>3}. {str(r['holder'])[:40]:<40} {r['total_pct_float']:.2f}%  "
          f"(cum: {r['cumulative_pct']:.2f}%)  {fmt_shares(r['total_shares'])}")

# Summary lines
for n in [5, 10, 20]:
    top_n = q12[q12['rank'] <= n]
    if len(top_n) > 0:
        print(f"\n  Top {n}: {top_n['total_pct_float'].sum():.2f}% of float")

q12.to_excel(f'../outputs/query12_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query12_{TICKER}_{today}.xlsx')

=== CONCENTRATION ANALYSIS: AR (Q4 2025) ===

    1. Vanguard Group                           9.86%  (cum: 9.86%)  28.7M
    2. BlackRock / iShares                      8.98%  (cum: 18.84%)  26.1M
    3. Fidelity / FMR                           8.14%  (cum: 26.98%)  23.7M
    4. Wellington Management                    3.89%  (cum: 30.87%)  11.3M
    5. State Street / SSGA                      3.40%  (cum: 34.27%)  9.9M
    6. Sourcerock Group LLC                     3.15%  (cum: 37.42%)  9.2M
    7. Dimensional Fund Advisors                3.13%  (cum: 40.55%)  9.1M
    8. Geode Capital Management                 1.84%  (cum: 42.39%)  5.3M
    9. Invesco                                  1.72%  (cum: 44.11%)  5.0M
   10. AQR Capital                              1.61%  (cum: 45.72%)  4.7M
   11. RAYMOND JAMES FINANCIAL INC              1.50%  (cum: 47.22%)  4.3M
   12. Goldman Sachs Asset Management           1.44%  (cum: 48.66%)  4.2M
   13. DE Shaw                                  1.4

---
## QUERY 13 — Energy Sector Institutional Rotation
Cross-name rotation among energy stocks by active managers.

In [14]:
# Shows which energy stocks active managers are rotating into/out of

q13 = con.execute("""
    WITH energy_moves AS (
        SELECT
            h4.ticker,
            h4.issuer_name,
            COUNT(DISTINCT CASE WHEN h4.shares > COALESCE(h1.shares, 0) THEN h4.cik END) as buyers,
            COUNT(DISTINCT CASE WHEN h4.shares < COALESCE(h1.shares, 0) THEN h4.cik END) as sellers,
            COUNT(DISTINCT CASE WHEN h1.cik IS NULL THEN h4.cik END) as new_positions,
            SUM(h4.market_value_live) as q4_total_value
        FROM holdings h4
        INNER JOIN securities s ON h4.cusip = s.cusip AND s.is_energy = true
        LEFT JOIN holdings h1 ON h4.cik = h1.cik AND h4.ticker = h1.ticker AND h1.quarter = '2025Q1'
        WHERE h4.quarter = '2025Q4'
          AND h4.manager_type IN ('active', 'hedge_fund', 'activist')
        GROUP BY h4.ticker, h4.issuer_name
    )
    SELECT *,
        buyers - sellers as net_flow,
        ROUND(buyers * 100.0 / (buyers + sellers), 1) as buy_pct
    FROM energy_moves
    WHERE buyers + sellers >= 5
    ORDER BY net_flow DESC
    LIMIT 25
""").fetchdf()

print('=== ENERGY SECTOR INSTITUTIONAL ROTATION (Q1→Q4 2025) ===')
print('Active/hedge fund managers only\n')
print(f'{"Ticker":<8} {"Issuer":<25} {"Buyers":>7} {"Sellers":>8} {"New":>5} {"Net":>5} {"Buy%":>6} {"Q4 Value":>12}')
print('-' * 85)
for _, r in q13.iterrows():
    print(f"{str(r['ticker'] or ''):<8} {str(r['issuer_name'])[:25]:<25} "
          f"{r['buyers']:>7} {r['sellers']:>8} {r['new_positions']:>5} "
          f"{r['net_flow']:>+5} {r['buy_pct']:>5.0f}% {fmt_dollars(r['q4_total_value']):>12}")

q13.to_excel(f'../outputs/query13_energy_rotation_{today}.xlsx', index=False)
print(f'\nExported to outputs/query13_energy_rotation_{today}.xlsx')

=== ENERGY SECTOR INSTITUTIONAL ROTATION (Q1→Q4 2025) ===
Active/hedge fund managers only

Ticker   Issuer                     Buyers  Sellers   New   Net   Buy%     Q4 Value
-------------------------------------------------------------------------------------
XOM      EXXON MOBIL CORP              317      241    46   +76    57%      $227.6B
ET       ENERGY TRANSFER L P           102       36    22   +66    74%        $2.4B
OKE      ONEOK INC NEW                 110       58    30   +52    66%        $3.5B
EQT      EQT CORP                       98       47    40   +51    68%        $8.8B
WMB      WILLIAMS COS INC              154      103    22   +51    60%       $38.3B
EXE      EXPAND ENERGY CORPORATION      73       26    24   +47    74%       $11.9B
KMI      KINDER MORGAN INC DEL         114       70    25   +44    62%        $4.5B
PSX      PHILLIPS 66                   143      101    42   +42    59%       $17.4B
CRC      CANADIAN NAT RES LTD           62       26    17   +36    

---
## QUERY 14 — Manager AUM vs Position Size Scatter Data
For a given stock, plot manager total AUM against their position size.

In [15]:
# ===== CHANGE TICKER HERE =====
TICKER = 'AR'
# ==============================

q14 = con.execute(f"""
    SELECT
        h.manager_name,
        h.manager_type,
        h.is_activist,
        m.aum_total / 1e9 as manager_aum_bn,
        h.market_value_live / 1e6 as position_mm,
        h.pct_of_portfolio,
        h.shares
    FROM holdings h
    LEFT JOIN managers m ON h.cik = m.cik
    WHERE h.ticker = '{TICKER}' AND h.quarter = '2025Q4'
      AND m.aum_total IS NOT NULL AND m.aum_total > 0
    ORDER BY h.market_value_live DESC NULLS LAST
    LIMIT 50
""").fetchdf()

print(f'=== MANAGER AUM vs POSITION SIZE: {TICKER} ===\n')
print(f'{"Manager":<40} {"AUM ($B)":>10} {"Position ($M)":>14} {"% Port":>8} {"Type":>12}')
print('-' * 90)
for _, r in q14.iterrows():
    flag = ' *' if r['is_activist'] else ''
    print(f"{str(r['manager_name'])[:40]:<40} {r['manager_aum_bn']:>10.1f} "
          f"{r['position_mm']:>14.1f} {fmt_pct(r['pct_of_portfolio']):>8} {str(r['manager_type']):>12}{flag}")

q14.to_excel(f'../outputs/query14_{TICKER}_{today}.xlsx', index=False)
print(f'\nExported to outputs/query14_{TICKER}_{today}.xlsx')

=== MANAGER AUM vs POSITION SIZE: AR ===

Manager                                    AUM ($B)  Position ($M)   % Port         Type
------------------------------------------------------------------------------------------
WELLINGTON MANAGEMENT GROUP LLP              1305.7          405.9    0.06%       active
DIMENSIONAL FUND ADVISORS LP                  835.7          357.6    0.06%      passive
MORGAN STANLEY                               1650.0           58.9    0.00%        mixed
WELLINGTON MANAGEMENT GROUP LLP              1305.7           52.1    0.01%       active
MILLENNIUM MANAGEMENT LLC                     571.1           48.7    0.02% quantitative
VICTORY CAPITAL MANAGEMENT INC                152.3           47.0    0.02%   hedge_fund
GW&K Investment Management, LLC                52.9           39.4    0.28%   hedge_fund
LOOMIS SAYLES & CO L P                        350.9           35.3    0.03%   hedge_fund
FIRST TRUST ADVISORS LP                       256.3           33.3

---
## QUERY 15 — Full Database Statistics
Summary stats for the entire 13-F database.

In [16]:
print('=== DATABASE STATISTICS ===\n')

stats = {
    'Total holdings rows': con.execute('SELECT COUNT(*) FROM holdings').fetchone()[0],
    'Unique filers (CIK)': con.execute('SELECT COUNT(DISTINCT cik) FROM holdings').fetchone()[0],
    'Unique securities (CUSIP)': con.execute('SELECT COUNT(DISTINCT cusip) FROM holdings').fetchone()[0],
    'Quarters loaded': con.execute('SELECT COUNT(DISTINCT quarter) FROM holdings').fetchone()[0],
    'Manager records': con.execute('SELECT COUNT(*) FROM managers').fetchone()[0],
    'Securities mapped': con.execute('SELECT COUNT(*) FROM securities').fetchone()[0],
    'Market data tickers': con.execute('SELECT COUNT(*) FROM market_data').fetchone()[0],
    'ADV records': con.execute('SELECT COUNT(*) FROM adv_managers').fetchone()[0],
}

for k, v in stats.items():
    print(f'  {k:<30} {v:>12,}')

print('\n--- Holdings by Quarter ---')
qstats = con.execute("""
    SELECT quarter, COUNT(*) as rows, COUNT(DISTINCT cik) as filers,
           COUNT(DISTINCT cusip) as securities,
           SUM(market_value_usd) / 1e12 as total_value_tn
    FROM holdings GROUP BY quarter ORDER BY quarter
""").fetchdf()
print(qstats.to_string(index=False))

print('\n--- Coverage Rates ---')
coverage = con.execute("""
    SELECT
        COUNT(*) as total,
        COUNT(CASE WHEN ticker IS NOT NULL THEN 1 END) as with_ticker,
        COUNT(CASE WHEN manager_type IS NOT NULL THEN 1 END) as with_manager_type,
        COUNT(CASE WHEN market_value_live IS NOT NULL THEN 1 END) as with_live_value,
        COUNT(CASE WHEN pct_of_float IS NOT NULL THEN 1 END) as with_float_pct
    FROM holdings WHERE quarter = '2025Q4'
""").fetchone()
total = coverage[0]
print(f'  Ticker coverage:      {coverage[1]:>8,} / {total:,} ({coverage[1]/total*100:.1f}%)')
print(f'  Manager type:         {coverage[2]:>8,} / {total:,} ({coverage[2]/total*100:.1f}%)')
print(f'  Live market value:    {coverage[3]:>8,} / {total:,} ({coverage[3]/total*100:.1f}%)')
print(f'  Float percentage:     {coverage[4]:>8,} / {total:,} ({coverage[4]/total*100:.1f}%)')

=== DATABASE STATISTICS ===

  Total holdings rows              12,270,984
  Unique filers (CIK)                   9,121
  Unique securities (CUSIP)            44,929
  Quarters loaded                           4
  Manager records                      12,017
  Securities mapped                    44,929
  Market data tickers                   5,993
  ADV records                          16,606

--- Holdings by Quarter ---
quarter    rows  filers  securities  total_value_tn
 2025Q1 2993162    8006       36558       51,780.24
 2025Q2 3047474    8060       35416       59,280.28
 2025Q3 3024698    8033       33283       64,985.08
 2025Q4 3205650    8636       33361       67,321.20

--- Coverage Rates ---
  Ticker coverage:      2,930,749 / 3,205,650 (91.4%)
  Manager type:          919,473 / 3,205,650 (28.7%)
  Live market value:    2,853,174 / 3,205,650 (89.0%)
  Float percentage:     2,223,650 / 3,205,650 (69.4%)
